# Flexible Cat Reward Experiment

This notebook runs a small, self-contained reward-design experiment without editing the original challenge notebook. Every run writes artifacts into a timestamped folder under `reward_function_experiments/results/`.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

import jax.numpy as jnp
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "reward_function_experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from reward_function_experiments.flexible_cat_reward import (
    cat_lifetime_loss,
    create_run_dir,
    default_reward_config,
    plot_metrics_history,
    save_metrics_history,
    save_run_config,
    save_summary,
    unpack_knobs,
)

run_dir = create_run_dir(PROJECT_ROOT / "reward_function_experiments" / "results", label="flexible_reward")
print(f"Run directory: {run_dir}")

## Configure Reward

The default reward is `log(Tx) + log(Tz) - lambda_bias * log(bias / eta_target)^2`. Optional alpha, photon-number, and parity terms are disabled by default. Flip the booleans below to try them.

In [ ]:
config = default_reward_config()
config.update(
    {
        "eta_target": 100.0,
        "lambda_bias": 2.0,
        "use_alpha_penalty": False,
        "alpha_target": 2.0,
        "lambda_alpha": 0.1,
        "use_nbar_penalty": False,
        "nbar_max": 8.0,
        "lambda_nbar": 0.1,
        "use_parity_bonus": False,
        "lambda_parity": 0.1,
        "tfinal_z": 200.0,
        "tfinal_x": 2.0,
        "na": 15,
        "nb": 5,
        "kappa_a": 1.0,
        "kappa_b": 10.0,
    }
)

save_run_config(config, run_dir)
config

## Smoke Test

In [ ]:
x0 = jnp.array([1.0, 0.0, 4.0, 0.0])
g2, eps_d = unpack_knobs(x0)
loss, metrics = cat_lifetime_loss(x0, config=config, return_metrics=True)

print("g2 =", g2)
print("eps_d =", eps_d)
print("loss =", loss)
print("metrics =", metrics)

## Small NumPy Evolutionary Optimizer

In [ ]:
POPULATION_SIZE = 4
N_GENERATIONS = 3
sigma0 = 0.25

lower_bounds = np.array([0.1, -1.0, 0.1, -2.0], dtype=float)
upper_bounds = np.array([3.0, 1.0, 8.0, 2.0], dtype=float)

rng = np.random.default_rng(2026)
mean = np.asarray(x0, dtype=float)
sigma = float(sigma0)
history = []
best = None

for generation in range(N_GENERATIONS):
    candidates = mean + rng.normal(0.0, sigma, size=(POPULATION_SIZE, 4))
    candidates[0] = mean
    candidates = np.clip(candidates, lower_bounds, upper_bounds)
    generation_rows = []

    for candidate_index, x_candidate in enumerate(candidates):
        loss, metrics = cat_lifetime_loss(jnp.asarray(x_candidate), config=config, return_metrics=True)
        row = {
            "generation": generation,
            "candidate_index": candidate_index,
            "x": np.asarray(x_candidate, dtype=float),
            "loss": float(loss),
            "metrics": metrics,
        }
        history.append(row)
        generation_rows.append(row)
        if best is None or float(loss) < float(best["loss"]):
            best = row

    generation_rows = sorted(generation_rows, key=lambda item: float(item["loss"]))
    elite_count = max(1, POPULATION_SIZE // 2)
    elite_mean = np.mean([row["x"] for row in generation_rows[:elite_count]], axis=0)
    mean = np.clip(0.7 * mean + 0.3 * elite_mean, lower_bounds, upper_bounds)
    sigma *= 0.85

    print(
        f"generation={generation} best_loss={best['loss']:.6g} "
        f"best_reward={best['metrics'].get('reward', np.nan):.6g}"
    )

history_path = save_metrics_history(history, run_dir)
plot_paths = plot_metrics_history(history, run_dir)

summary = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "best_x": best["x"].tolist(),
    "best_loss": float(best["loss"]),
    "best_reward": float(best["metrics"].get("reward", np.nan)),
    "best_metrics": best["metrics"],
    "population_size": POPULATION_SIZE,
    "n_generations": N_GENERATIONS,
    "run_dir": str(run_dir),
}
summary_path = save_summary(summary, run_dir)

print("history:", history_path)
print("summary:", summary_path)
print("plots:")
for path in plot_paths:
    print(" -", path)